# AI Analysis Module

Generates grounded, data-referenced natural language analysis using one of three LLM backends.
Change `BACKEND` in the setup cell to switch.

| Backend | Model | Cost | Setup |
|---------|-------|------|-------|
| `"gemini"` | Gemini 2.0 Flash | Free (15 RPM) | `GEMINI_API_KEY` in `.env` — [get key](https://aistudio.google.com) |
| `"groq"` | Llama 3.3 70B | Free (14,400 req/day) | `GROQ_API_KEY` in `.env` — [get key](https://console.groq.com) |
| `"ollama"` | llama3.2 (local) | **Free, unlimited** | Install [Ollama](https://ollama.com) → `ollama pull llama3.2` |

Generates four analysis sections:
1. **Trend & Performance** — current trend and 90-day performance per asset
2. **Anomaly Detection** — largest single-day moves and likely causes
3. **Risk Commentary** — volatility-based risk ranking and portfolio risk note
4. **Cross-Asset Comparison** — relative performance and correlation insights

Full report exported to `outputs/ai_analysis_report.md`.


In [15]:
!pip install google-generativeai
!pip install groq
!pip install ollama
!pip install python-dotenv
import pandas as pd
import numpy as np
import json
import time
import os
import google.generativeai as genai
from groq import Groq
import ollama as _ollama
from dotenv import load_dotenv

load_dotenv()   # reads API keys from .env

# ── Backend selector ────────────────────────────────────────────────────────
# "gemini" → Google Gemini 2.0 Flash  (free: 15 RPM — may hit quota)
# "groq"   → Groq Llama 3.3 70B       (free: 14,400 req/day, fast)
# "ollama" → Local model via Ollama    (no API key, no rate limits)
#             Requires: ollama.com installed + `ollama pull llama3.2`
BACKEND      = "groq"
OLLAMA_MODEL = "llama3.2"   # only used when BACKEND = "ollama"

if BACKEND == "gemini":
    genai.configure(api_key=os.environ["GEMINI_API_KEY"])
    _gemini_model = genai.GenerativeModel("gemini-2.0-flash")
    MODEL_LABEL = "Google Gemini 2.0 Flash"
elif BACKEND == "groq":
    _groq_client = Groq(api_key="PASTE_API_KEY_HERE")
    MODEL_LABEL = "Groq — Llama 3.3 70B"
elif BACKEND == "ollama":
    MODEL_LABEL = f"Ollama — {OLLAMA_MODEL} (local)"
else:
    raise ValueError(f"Unknown BACKEND: {BACKEND!r}. Use 'gemini', 'groq', or 'ollama'.")

FOCUS_TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "JPM", "JNJ"]

os.makedirs("outputs", exist_ok=True)

print("AI Analysis Module")
print("=" * 50)
print(f"Backend : {MODEL_LABEL}")
print(f"Assets  : {FOCUS_TICKERS}")


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


AI Analysis Module
Backend : Groq — Llama 3.3 70B
Assets  : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM', 'JNJ']


## Load Data & Compute Summary Statistics

In [16]:
price_df = pd.read_csv("data/cleaned/prices_clean.csv", parse_dates=["Date"])
price_df = price_df[price_df["Ticker"].isin(FOCUS_TICKERS)].copy()
price_df = price_df.sort_values(["Ticker", "Date"])

# ── 90-day summary stats per ticker ─────────────────────────────────────────
cutoff  = price_df["Date"].max() - pd.Timedelta(days=90)
recent  = price_df[price_df["Date"] >= cutoff].copy()

def _stats(group):
    g = group.sort_values("Date")
    latest, first = g.iloc[-1], g.iloc[0]

    close_latest = float(latest["Close"])
    close_first  = float(first["Close"])
    ret_90d      = (close_latest / close_first - 1) * 100 if close_first else None

    vol_series = g["volatility_30d"].dropna()
    vol_latest = float(vol_series.iloc[-1]) * 100 if not vol_series.empty else None

    return pd.Series({
        "latest_date"             : str(latest["Date"].date()),
        "latest_close"            : round(close_latest, 2),
        "return_90d_pct"          : round(ret_90d, 2) if ret_90d is not None else None,
        "volatility_30d_ann_pct"  : round(vol_latest, 2) if vol_latest is not None else None,
        "ma7_latest"              : round(float(latest["ma7"]), 2) if pd.notna(latest["ma7"]) else None,
        "ma30_latest"             : round(float(latest["ma30"]), 2) if pd.notna(latest["ma30"]) else None,
        "above_ma30"              : bool(close_latest > float(latest["ma30"])) if pd.notna(latest["ma30"]) else None,
        "max_daily_loss_pct"      : round(float(g["daily_return"].min()) * 100, 2),
        "max_daily_gain_pct"      : round(float(g["daily_return"].max()) * 100, 2),
        "avg_daily_return_pct"    : round(float(g["daily_return"].mean()) * 100, 4),
        "n_outlier_days"          : int(g["is_outlier"].sum()) if "is_outlier" in g.columns else 0,
    })

stats_df = recent.groupby("Ticker").apply(_stats).reset_index()
print("90-day summary statistics:")
stats_df

90-day summary statistics:


,Ticker,latest_date,latest_close,return_90d_pct,volatility_30d_ann_pct,ma7_latest,ma30_latest,above_ma30,max_daily_loss_pct,max_daily_gain_pct,avg_daily_return_pct,n_outlier_days
0,AAPL,2025-12-31,271.36,5.83,13.05,272.65,275.39,False,-3.45,3.94,0.1031,0
1,AMZN,2025-12-31,230.82,3.78,19.51,232.12,228.72,True,-4.99,9.58,0.0897,0
2,GOOGL,2025-12-31,312.78,27.48,28.82,313.56,312.38,True,-3.21,6.31,0.3961,0
3,JNJ,2025-12-31,205.86,11.98,16.21,206.10,205.38,True,-2.27,3.29,0.1780,0
4,JPM,2025-12-31,319.14,5.28,23.61,322.82,311.43,True,-4.66,3.19,0.0731,0
5,META,2025-12-31,659.53,-9.14,23.56,663.44,648.09,True,-11.33,3.78,-0.1048,0
6,MSFT,2025-12-31,481.48,-6.05,17.47,484.81,481.11,True,-2.92,2.17,-0.1010,0
7,NVDA,2025-12-31,186.49,-1.26,29.74,188.45,181.90,True,-4.89,5.79,0.0196,0


## LLM Helper

In [17]:
SYSTEM_PROMPT = """You are a professional financial analyst assistant.
Your task is to produce concise, data-grounded analysis of financial assets.
Always reference specific numbers from the data provided.
Do NOT speculate beyond what the data shows.
Use professional financial language suitable for a fintech report."""


def call_llm(user_prompt: str, max_tokens: int = 900) -> str:
    """Call the selected LLM backend with retry logic (up to 3 attempts)."""
    full_prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

    for attempt in range(3):
        try:
            if BACKEND == "gemini":
                response = _gemini_model.generate_content(
                    full_prompt,
                    generation_config=genai.GenerationConfig(
                        max_output_tokens=max_tokens,
                        temperature=0.3,
                    ),
                )
                return response.text.strip()

            elif BACKEND == "groq":
                response = _groq_client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_prompt},
                    ],
                    max_tokens=max_tokens,
                    temperature=0.3,
                )
                return response.choices[0].message.content.strip()

            elif BACKEND == "ollama":
                response = _ollama.chat(
                    model=OLLAMA_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_prompt},
                    ],
                    options={"temperature": 0.3, "num_predict": max_tokens},
                )
                return response["message"]["content"].strip()

        except Exception as e:
            print(f"  [Attempt {attempt + 1}] Error: {e}")
            if attempt < 2:
                time.sleep(5)

    return "[Error: LLM call failed after 3 attempts]"


print(f"LLM helper ready — backend: {MODEL_LABEL}")

LLM helper ready — backend: Groq — Llama 3.3 70B


## Analysis 1 — Trend & Performance Summary

In [18]:
print("=== Trend & Performance Analysis ===\n")

stats_json = stats_df.set_index("Ticker").to_dict(orient="index")

trend_prompt = f"""Analyse the recent 90-day performance of the following financial assets.

For EACH asset, write 2–3 sentences covering:
- Current price vs 30-day moving average (uptrend / downtrend signal)
- 90-day percentage return
- Whether MA7 crossing above/below MA30 signals momentum

Data (JSON, last 90 days):
{json.dumps(stats_json, indent=2)}

Format your response as a bullet list (one entry per ticker, label each with the ticker symbol).
Reference specific prices and percentages in every entry."""

trend_analysis = call_llm(trend_prompt, max_tokens=1200)
print(trend_analysis)

=== Trend & Performance Analysis ===

  [Attempt 1] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
  [Attempt 2] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
  [Attempt 3] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
[Error: LLM call failed after 3 attempts]


## Analysis 2 — Anomaly & Notable Event Detection

In [19]:
print("=== Anomaly & Notable Event Detection ===\n")

# Top 3 largest daily moves per ticker
anomaly_records = []
for ticker in FOCUS_TICKERS:
    t_df = price_df[price_df["Ticker"] == ticker].copy()
    t_df["abs_return"] = t_df["daily_return"].abs()
    top3 = t_df.nlargest(3, "abs_return")[["Date", "daily_return", "Close"]]
    for _, row in top3.iterrows():
        anomaly_records.append({
            "ticker"          : ticker,
            "date"            : str(row["Date"].date()),
            "daily_return_pct": round(float(row["daily_return"]) * 100, 2),
            "close_price"     : round(float(row["Close"]), 2),
        })

# Sort by magnitude
anomaly_records.sort(key=lambda x: abs(x["daily_return_pct"]), reverse=True)

anomaly_prompt = f"""The following table lists the largest single-day price movements detected
in the dataset across a set of S&P 500 assets.

Data (JSON — sorted by absolute daily return):
{json.dumps(anomaly_records, indent=2)}

Write a professional summary (4–5 sentences) that:
1. Identifies which tickers showed the most extreme movements and on which dates
2. Notes which moves are likely genuine market events vs possible data anomalies (>30% in one day)
3. Speculates on likely catalysts (earnings surprises, macro events, sector rotation)
4. States which ticker appears most prone to large single-day swings overall"""

anomaly_analysis = call_llm(anomaly_prompt, max_tokens=700)
print(anomaly_analysis)

=== Anomaly & Notable Event Detection ===

  [Attempt 1] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
  [Attempt 2] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
  [Attempt 3] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
[Error: LLM call failed after 3 attempts]


## Analysis 3 — Risk Commentary

In [20]:
print("=== Risk Commentary ===\n")

vol_table = (
    stats_df[["Ticker", "volatility_30d_ann_pct", "return_90d_pct",
              "max_daily_loss_pct", "avg_daily_return_pct"]]
    .sort_values("volatility_30d_ann_pct", ascending=False)
)

risk_prompt = f"""Based on the following volatility and risk metrics for a portfolio of assets,
write a professional risk commentary.

Volatility & risk data (sorted high→low by annualised 30-day volatility):
{vol_table.to_string(index=False)}

Your commentary must:
1. Rank assets from highest to lowest risk and explain the ranking
2. Comment on risk-adjusted return (return per unit of volatility) for each asset
3. Identify which assets appear high-risk/high-reward vs defensive/low-volatility
4. Provide a brief portfolio-level risk observation if an investor held all these assets equally
5. Reference specific numbers (volatility %, return %) throughout

Keep response to 5–7 sentences."""

risk_analysis = call_llm(risk_prompt, max_tokens=800)
print(risk_analysis)

=== Risk Commentary ===

  [Attempt 1] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
  [Attempt 2] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
  [Attempt 3] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
[Error: LLM call failed after 3 attempts]


## Analysis 4 — Cross-Asset Comparison

In [21]:
print("=== Cross-Asset Comparison ===\n")

# Correlation matrix on last 12 months of daily returns
cutoff_12m = price_df["Date"].max() - pd.Timedelta(days=365)
returns_wide = (
    price_df[price_df["Date"] >= cutoff_12m]
    .pivot_table(index="Date", columns="Ticker", values="daily_return")
    .dropna(how="all")
)

# Top 8 highest-correlation pairs
corr_pairs = []
tl = [t for t in FOCUS_TICKERS if t in returns_wide.columns]
for i, t1 in enumerate(tl):
    for t2 in tl[i + 1:]:
        pair_corr = returns_wide[[t1, t2]].dropna().corr().iloc[0, 1]
        corr_pairs.append({"pair": f"{t1}/{t2}", "correlation": round(pair_corr, 3)})
corr_pairs.sort(key=lambda x: abs(x["correlation"]), reverse=True)

comparison_prompt = f"""Compare the following financial assets based on their 90-day performance
and 12-month correlation data.

Performance summary (90 days):
{stats_df[['Ticker','return_90d_pct','volatility_30d_ann_pct','above_ma30']].to_string(index=False)}

Top correlated pairs (last 12 months, by absolute correlation):
{json.dumps(corr_pairs[:10], indent=2)}

Write a 4–5 sentence comparison paragraph that:
1. Compares the best and worst performing assets over the 90-day period
2. Notes which pairs move most closely together and likely reasons (same sector, macro sensitivity)
3. Highlights any notable performance divergence within the group
4. Concludes with which asset delivered the best risk-adjusted performance

Reference specific tickers, percentages, and correlation values throughout."""

comparison_analysis = call_llm(comparison_prompt, max_tokens=800)
print(comparison_analysis)

=== Cross-Asset Comparison ===

  [Attempt 1] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
  [Attempt 2] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
  [Attempt 3] Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
[Error: LLM call failed after 3 attempts]


## Export Full Report

In [22]:
report_path = "outputs/ai_analysis_report.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# AI Financial Analysis Report\n\n")
    f.write(f"**Generated :** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n")
    f.write(f"**Model     :** {MODEL_LABEL}\n")
    f.write(f"**Assets    :** {', '.join(FOCUS_TICKERS)}\n\n")
    f.write("---\n\n")

    f.write("## 1. Trend & Performance Analysis\n\n")
    f.write(trend_analysis + "\n\n")

    f.write("## 2. Anomaly & Notable Events\n\n")
    f.write(anomaly_analysis + "\n\n")

    f.write("## 3. Risk Commentary\n\n")
    f.write(risk_analysis + "\n\n")

    f.write("## 4. Cross-Asset Comparison\n\n")
    f.write(comparison_analysis + "\n\n")

    f.write("---\n")
    f.write("*This report was generated automatically. "
            "It is based on historical data and does not constitute financial advice.*\n")

print(f"Report saved to: {report_path}")
print(f"File size: {os.path.getsize(report_path):,} bytes")

Report saved to: outputs/ai_analysis_report.md
File size: 611 bytes
